# Free Ray multi-agent lab
Use a free CPU runtime with Python 3.10–3.12. Upload a zip of this repository (GitHub → Code → Download ZIP). No API keys or paid APIs are used.
The personas are deterministic simulations, not Gemma. Colab does not expose a private localhost dashboard to your laptop; use the JSON output here and the local/GCP workflow for dashboard screenshots.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, sys, subprocess, os
assert (3, 10) <= sys.version_info[:2] < (3, 13), 'Select a Python 3.10–3.12 runtime'
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
workspace = Path('/content/ray-lab')
workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    bundle.extractall(workspace)
project = next(path.parent for path in workspace.rglob('pyproject.toml') if (path.parent / 'src/gemma_ray').is_dir())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(project)], check=True)
os.chdir(project)
os.environ['RAY_USAGE_STATS_ENABLED'] = '0'

In [ ]:
# A separate driver process makes rerunning the cell cleanly start/stop Ray.
subprocess.run([sys.executable, '-m', 'gemma_ray.cli', '--jobs', '4'], check=True)
import json
report = json.loads(Path('artifacts/run.json').read_text())
[(item['research']['topic'], item['approved']) for item in report['results']]

In [ ]:
# Inspect shared context and each persona's worker process.
print('Shared context:', report['context']['context_id'])
for persona, events in report['actor_events'].items():
    print(persona, 'PID:', events[0]['pid'], 'messages:', len(events))
files.download('artifacts/run.json')

Try live tools by adding repeated `--url` arguments to the CLI subprocess, such as `https://api.github.com/repos/ray-project/ray`. Network/rate-limit failures are recorded in the report. See the README for object-store explanations and the separate bounded GCP session.